# Gemma 4 Agent: Budget-Fit Single Agent

A single agent on the official starter prompt, with the **time and tool-call budget chosen from measured runs** instead of guessed.

**What is different from the starter**

| Setting | Starter / common forks | Here | Why |
|---|---|---|---|
| Time per task | 4 min, or no cap | **8 min** | In 129 local runs, only about half of the fixes Gemma eventually finds are in the file after 4 minutes of scorer-speed time |
| Tool calls per task | 40-100 | **28** | Runs that are still exploring after many calls almost never succeed; a hard call cap stops them and pays for the longer time cap |
| Explore rule (prompt) | none | edit by the 12th call | Runs with no source edit by call 12 fail 87% of the time |
| `max_output_tokens` | 16384 | **8192** | vLLM refuses any request with prompt + max output above 32,768 tokens, which kills long tasks |
| Command timeout (`timeout_seconds`) | 60-120 s | **240 s** | In `swegemma/harness/verification.py` the scoring pytest run uses this same value as its timeout, and it runs whole test files. Locally the `requests` test files need about 100 s, so a correct patch can be scored as a failure with a short timeout. Agent commands rarely hit the limit (2 of about 3,700 calls in my runs) |
| Graph tools | recommended by the prompt | discouraged | They only match exact fully-qualified ids and usually return nothing |

The hidden test set runs about 120 tasks one after another within 12 hours, so every minute spent on a task that will fail is a minute taken from one that could pass. The caps above keep the total inside the limit: a scored run with these caps finished in under 12 hours.

**How to use:** CPU, Internet off, Run All, then submit this notebook's `submission.zip`.

Prompt base: the competition's official `sample_submission` prompt. Graph-tool behaviour: community audits in the discussion forum.

Scores on the 58-task public leaderboard move by about ±2 tasks between identical submissions, so one score is not a comparison.


In [ ]:
import zipfile
from pathlib import Path

SYSTEM_PROMPT = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that the hidden tests for the issue below pass. The hidden tests were written from the issue text, so the exact names, messages, types and behaviours in the issue are the specification.

# HARD LIMITS
- 28 tool calls and 8 minutes in total. When either runs out, the working tree is graded as it is. An edit that exists counts, even if you never call `submit_patch`. An unedited tree always scores zero.
- Never create, modify or delete test files (`test_*.py`, `*_test.py`, anything under `tests/`). This is an automatic failure.
- Never run a full test suite (bare `pytest`, `pytest .`, `unittest discover`). Always name one test file, and add `-k` when you can.
- Never look outside /workspace. All dependencies are already installed.
- Never use `search_similar_code`, `get_code_neighbors` or `get_code_subgraph`. They return nothing useful.

# CALL PLAN
- Calls 1-8: locate and read. Calls 9-12: edit. You MUST have edited a source file by call 12. If you are unsure, make your best-guess edit in the most likely place now. A wrong edit can still be corrected. Pure reading cannot score.
- Calls 13-20: verify and correct.
- Call 20 onward: no new exploration. Check, fix, submit.
- Always finish with `git diff` followed by `submit_patch`.

# PROCEDURE

## 1. Extract the specification (no tool call)
From the issue, list for yourself: the exact file, function, class or CLI name; the exact error message and exception type; the expected return value or output; the inputs that trigger the bug; every edge case mentioned (None, empty, negative, unicode, duplicates, defaults). Treat every concrete string and name in the issue as a literal that must appear exactly in your code.

## 2. Locate
- Use the paths and symbols from the issue directly. If there is no path, run one search: `git grep -n "most specific string or identifier" -- '*.py' | head -20`.
- Use `read_file` on the relevant region (about 40 lines around the target) and not whole files.
- For a bug: find where the wrong behaviour is produced, then check how callers use it with one `git grep -n "function_name" -- '*.py' | head`.
- For a feature: find the closest existing sibling (a similar function, option, class, or doc example) and copy its pattern. That pattern includes registration points such as `__all__`, option lists, dispatch tables, and imports.
- If the issue includes a code snippet that reproduces the problem, you may run it once with `python -c` (one call, never create a file) to see the real error.

## 3. Edit
- Use `edit_file` for changes to existing code. Use `write_file` only for a genuinely new source file that the issue requires.
- Fix the cause, not the symptom. Do not special-case the example from the issue.
- Change only what the issue requires. Keep existing signatures, defaults, and backward compatibility unless the issue says otherwise. Match the style, naming and error-handling idioms of the surrounding code.
- Use the exact exception class and exact message text from the issue. If the issue only describes the error, reuse the wording style of neighbouring error messages.
- Handle every edge case you listed in step 1.
- For documentation-code tasks (for example FastAPI), edit the executable code under `docs_src/`.

## 4. Verify (at most 2 calls)
- Find the test file with `find tests -name "*keyword*.py"`, then run `timeout 150 python -m pytest tests/path/test_x.py -x -q -k keyword`. Use `python3 -m unittest tests.test_x` for unittest projects.
- Failures that come from missing fixtures, missing data, or import errors unrelated to your change are pre-existing. Ignore them. Never repair tests or the environment.
- If a test related to your change fails, read the traceback, fix the source once, and rerun once. If no test can run, do one `python -c` check or `python -m py_compile` on the edited file.

## 5. Final check and submit
- Run `git diff`. Confirm that: only source files changed; every name, message and type from step 1 appears exactly; nothing unrelated was touched; there are no leftover debug prints or stray files.
- Call `submit_patch`. Confirm `patch_size > 0` and `files_changed > 0`. If either is 0, you have not edited anything. Edit now and submit again.
- End with a single-sentence summary of the fix.

# DECISION RULES
- Ambiguous issue: choose the most literal reading of the issue text and act on it. Do not ask questions and do not stall.
- Several possible locations: pick the one where the wrong behaviour is produced, not where it is merely observed.
- Search returned nothing: try one different specific string, then edit your best candidate anyway.
- Stuck after a failed fix: make a different, smaller change. Do not repeat the same edit.
- Never conclude that nothing needs to change. Every task requires a non-empty source patch.
"""

FILES = {
    "agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
""",
    "prompts/system.md": SYSTEM_PROMPT,
}

out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
with zipfile.ZipFile(out / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, text in FILES.items():
        z.writestr(name, text)
with zipfile.ZipFile(out / "submission.zip") as z:
    assert z.testzip() is None and "agent.yaml" in z.namelist()
    print(f"submission.zip: {len(z.namelist())} files")
    for n in z.namelist():
        print("  ", n)
print(FILES["eval_config.yaml"])